In [12]:
import math
import os
import os.path as osp
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import torch
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib
import warnings
warnings.filterwarnings('ignore')

matplotlib.rc("font", family='AR PL UKai CN')


In [13]:
# 定义高斯分布embedding筛选器类
class GaussianEmbeddingFilter:
    """
    基于高斯分布的embedding筛选器
    
    为每个类别的embedding建立多元高斯分布模型，
    通过均值和协方差进行有效筛选，使用余弦相似度判断embedding相似性
    """
    
    def __init__(
        self,
        embeddings: np.ndarray,
        labels: np.ndarray,
        regularization: float = 1e-6
    ):
        """
        初始化筛选器
        
        Args:
            embeddings: 特征向量 (N, D)
            labels: 标签 (N,)
            regularization: 协方差矩阵正则化参数，防止奇异矩阵
        """
        self.embeddings = embeddings
        self.labels = np.array(labels)
        self.regularization = regularization
        self.unique_labels = np.unique(self.labels)
        
        # 为每个类别建立高斯分布模型
        self.gaussian_models = {}
        self._fit_gaussian_models()
    
    def _fit_gaussian_models(self):
        """为每个类别拟合多元高斯分布"""
        for label in self.unique_labels:
            mask = self.labels == label
            class_embeddings = self.embeddings[mask]
            
            if len(class_embeddings) < 2:
                # 样本数太少，无法估计协方差
                mean = class_embeddings[0] if len(class_embeddings) == 1 else np.zeros(self.embeddings.shape[1])
                cov = np.eye(self.embeddings.shape[1]) * self.regularization
            else:
                # 计算均值和协方差
                mean = np.mean(class_embeddings, axis=0)
                cov = np.cov(class_embeddings.T)
                
                # 正则化协方差矩阵，防止奇异
                cov += np.eye(cov.shape[0]) * self.regularization
            
            # 存储高斯模型参数
            self.gaussian_models[label] = {
                'mean': mean,
                'cov': cov,
                'n_samples': len(class_embeddings)
            }
    
    def compute_mahalanobis_distance(
        self,
        embeddings: np.ndarray,
        label: str
    ) -> np.ndarray:
        """计算embedding到指定类别高斯分布的马氏距离"""
        if label not in self.gaussian_models:
            raise ValueError(f"标签 {label} 不存在于模型中")
        
        model = self.gaussian_models[label]
        mean = model['mean']
        cov = model['cov']
        
        embeddings = np.atleast_2d(embeddings)
        diff = embeddings - mean
        cov_inv = np.linalg.inv(cov)
        mahalanobis_dist = np.sqrt(np.sum(diff @ cov_inv * diff, axis=1))
        
        return mahalanobis_dist[0] if embeddings.shape[0] == 1 else mahalanobis_dist
    
    def compute_cosine_similarity(
        self,
        embeddings1: np.ndarray,
        embeddings2: np.ndarray = None
    ) -> np.ndarray:
        """计算embedding之间的余弦相似度"""
        embeddings1 = np.atleast_2d(embeddings1)
        
        if embeddings2 is None:
            similarity = cosine_similarity(embeddings1)
        else:
            embeddings2 = np.atleast_2d(embeddings2)
            similarity = cosine_similarity(embeddings1, embeddings2)
        
        return similarity
    
    def filter_by_mahalanobis(
        self,
        embeddings: np.ndarray,
        labels: np.ndarray,
        threshold_percentile: float = 95.0,
        return_scores: bool = False
    ):
        """基于马氏距离筛选embedding"""
        distances = []
        valid_indices = []
        
        for i, (emb, label) in enumerate(zip(embeddings, labels)):
            try:
                dist = self.compute_mahalanobis_distance(emb, label)
                distances.append(dist)
                valid_indices.append(i)
            except:
                continue
        
        if not distances:
            if return_scores:
                return np.array([]), np.array([]), np.array([])
            return np.array([]), np.array([])
        
        distances = np.array(distances)
        threshold = np.percentile(distances, threshold_percentile)
        
        mask = distances <= threshold
        filtered_indices = np.array(valid_indices)[mask]
        
        filtered_embeddings = embeddings[filtered_indices]
        filtered_labels = labels[filtered_indices]
        filtered_distances = distances[mask]
        
        if return_scores:
            return filtered_embeddings, filtered_labels, filtered_distances
        return filtered_embeddings, filtered_labels
    
    def get_class_statistics(self, label: str):
        """获取指定类别的统计信息"""
        if label not in self.gaussian_models:
            raise ValueError(f"标签 {label} 不存在于模型中")
        
        model = self.gaussian_models[label]
        mask = self.labels == label
        class_embeddings = self.embeddings[mask]
        
        # 计算类内余弦相似度
        cosine_sim = self.compute_cosine_similarity(class_embeddings)
        mask_upper = np.triu(np.ones_like(cosine_sim, dtype=bool), k=1)
        intra_cosine_sim = cosine_sim[mask_upper]
        
        # 计算每个样本的马氏距离
        mahalanobis_dists = self.compute_mahalanobis_distance(class_embeddings, label)
        
        return {
            'label': label,
            'n_samples': model['n_samples'],
            'mean': model['mean'],
            'cov': model['cov'],
            'mean_mahalanobis_distance': np.mean(mahalanobis_dists),
            'std_mahalanobis_distance': np.std(mahalanobis_dists),
            'mean_intra_cosine_similarity': np.mean(intra_cosine_sim),
            'std_intra_cosine_similarity': np.std(intra_cosine_sim)
        }
    
    def analyze_all_classes(self):
        """分析所有类别的统计信息"""
        results = {}
        for label in self.unique_labels:
            results[label] = self.get_class_statistics(label)
        return results


In [14]:
# 加载数据
npz_path = "../data/zhenyu_data/embeddings/supcon_labels/1201_ssl_nopretrained.npz"
data = np.load(npz_path)
embeddings = data['embeddings']
instance_ids = data['instance_ids']
labels = data['labels']

print(f"原始数据形状: {embeddings.shape}")
print(f"标签数量: {len(labels)}")
print(f"唯一标签: {sorted(set(labels))}")


In [15]:
# 数据预处理：选择所有标签
selected_embeddings = []
selected_labels = []
for emb, lb in zip(embeddings, labels):
    selected_embeddings.append(emb)
    selected_labels.append(lb)

selected_embeddings = np.stack(selected_embeddings)
selected_labels = np.array(selected_labels)

print(f"选择的embeddings形状: {selected_embeddings.shape}")
print(f"选择的标签: {sorted(set(selected_labels))}")
print(f"每个标签的样本数:")
for label in sorted(set(selected_labels)):
    count = np.sum(selected_labels == label)
    print(f"  {label}: {count}")


In [16]:
# 初始化高斯分布筛选器
filter_model = GaussianEmbeddingFilter(
    embeddings=selected_embeddings,
    labels=selected_labels,
    regularization=1e-6
)

print(f"已为 {len(filter_model.unique_labels)} 个类别建立高斯分布模型")
print(f"类别列表: {sorted(filter_model.unique_labels)}")


In [17]:
# 分析所有类别的统计信息
stats = filter_model.analyze_all_classes()

print("=" * 80)
print("各类别统计信息")
print("=" * 80)

for label in sorted(stats.keys()):
    stat = stats[label]
    print(f"\n类别: {stat['label']}")
    print(f"  样本数: {stat['n_samples']}")
    print(f"  平均马氏距离: {stat['mean_mahalanobis_distance']:.4f} ± {stat['std_mahalanobis_distance']:.4f}")
    print(f"  类内平均余弦相似度: {stat['mean_intra_cosine_similarity']:.4f} ± {stat['std_intra_cosine_similarity']:.4f}")


In [18]:
# 计算每个样本的马氏距离
mahalanobis_distances = []

for i, (emb, label) in enumerate(zip(selected_embeddings, selected_labels)):
    maha_dist = filter_model.compute_mahalanobis_distance(emb, label)
    mahalanobis_distances.append(maha_dist)

mahalanobis_distances = np.array(mahalanobis_distances)

print(f"马氏距离统计:")
print(f"  均值: {np.mean(mahalanobis_distances):.4f}")
print(f"  标准差: {np.std(mahalanobis_distances):.4f}")
print(f"  最小值: {np.min(mahalanobis_distances):.4f}")
print(f"  最大值: {np.max(mahalanobis_distances):.4f}")
print(f"  95%分位数: {np.percentile(mahalanobis_distances, 95):.4f}")


In [19]:
# 基于马氏距离筛选（保留95%的样本，去除离群点）
filtered_emb_maha, filtered_labels_maha, maha_scores = filter_model.filter_by_mahalanobis(
    embeddings=selected_embeddings,
    labels=selected_labels,
    threshold_percentile=95.0,
    return_scores=True
)

print(f"马氏距离筛选结果:")
print(f"  原始样本数: {len(selected_embeddings)}")
print(f"  筛选后样本数: {len(filtered_emb_maha)}")
print(f"  保留率: {len(filtered_emb_maha) / len(selected_embeddings) * 100:.2f}%")
print(f"  阈值: {np.percentile(maha_scores, 95):.4f}")

# 统计每个类别筛选后的样本数
print(f"\n各类别筛选后样本数:")
for label in sorted(set(filtered_labels_maha)):
    count = np.sum(filtered_labels_maha == label)
    original_count = np.sum(selected_labels == label)
    print(f"  {label}: {count}/{original_count} ({count/original_count*100:.1f}%)")


In [20]:
# 计算embedding之间的余弦相似度矩阵
similarity_matrix = filter_model.compute_cosine_similarity(selected_embeddings)
print(f"相似度矩阵形状: {similarity_matrix.shape}")

# 计算每个类别内部的平均相似度
print(f"\n各类别内部余弦相似度:")
for label in sorted(filter_model.unique_labels):
    mask = selected_labels == label
    class_embeddings = selected_embeddings[mask]
    class_similarity = filter_model.compute_cosine_similarity(class_embeddings)
    # 去除对角线（自己与自己的相似度）
    mask_upper = np.triu(np.ones_like(class_similarity, dtype=bool), k=1)
    intra_similarity = class_similarity[mask_upper]
    print(f"  {label}: {np.mean(intra_similarity):.4f} ± {np.std(intra_similarity):.4f}")

# 计算类别之间的平均相似度
print(f"\n类别之间的余弦相似度:")
unique_labels_sorted = sorted(filter_model.unique_labels)
for i, label1 in enumerate(unique_labels_sorted):
    for j, label2 in enumerate(unique_labels_sorted):
        if i < j:
            mask1 = selected_labels == label1
            mask2 = selected_labels == label2
            emb1 = selected_embeddings[mask1]
            emb2 = selected_embeddings[mask2]
            inter_similarity = filter_model.compute_cosine_similarity(emb1, emb2)
            print(f"  {label1} <-> {label2}: {np.mean(inter_similarity):.4f} ± {np.std(inter_similarity):.4f}")


In [21]:
# 可视化：马氏距离分布
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. 马氏距离分布
axes[0].hist(mahalanobis_distances, bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(np.percentile(mahalanobis_distances, 95), color='r', linestyle='--', 
                label=f'95%分位数: {np.percentile(mahalanobis_distances, 95):.2f}')
axes[0].set_xlabel('马氏距离')
axes[0].set_ylabel('频数')
axes[0].set_title('马氏距离分布')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. 各类别马氏距离箱线图
maha_by_label = []
labels_for_box = []
for label in sorted(filter_model.unique_labels):
    mask = selected_labels == label
    maha_by_label.append(mahalanobis_distances[mask])
    labels_for_box.append(label)

axes[1].boxplot(maha_by_label, labels=labels_for_box)
axes[1].set_ylabel('马氏距离')
axes[1].set_title('各类别马氏距离分布')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [22]:
# 可视化：余弦相似度热力图
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. 整体相似度矩阵热力图
im1 = axes[0].imshow(similarity_matrix, cmap='viridis', aspect='auto', vmin=0, vmax=1)
axes[0].set_title('所有Embedding之间的余弦相似度矩阵')
axes[0].set_xlabel('样本索引')
axes[0].set_ylabel('样本索引')
plt.colorbar(im1, ax=axes[0])

# 2. 类别内部相似度统计
intra_similarities = []
inter_similarities = []

for i, label1 in enumerate(sorted(filter_model.unique_labels)):
    mask1 = selected_labels == label1
    emb1 = selected_embeddings[mask1]
    
    # 类内相似度
    sim1 = filter_model.compute_cosine_similarity(emb1)
    mask_upper = np.triu(np.ones_like(sim1, dtype=bool), k=1)
    intra_similarities.extend(sim1[mask_upper].tolist())
    
    # 类间相似度
    for j, label2 in enumerate(sorted(filter_model.unique_labels)):
        if i < j:
            mask2 = selected_labels == label2
            emb2 = selected_embeddings[mask2]
            sim2 = filter_model.compute_cosine_similarity(emb1, emb2)
            inter_similarities.extend(sim2.flatten().tolist())

axes[1].hist(intra_similarities, bins=50, alpha=0.7, label='类内相似度', color='blue')
axes[1].hist(inter_similarities, bins=50, alpha=0.7, label='类间相似度', color='red')
axes[1].set_xlabel('余弦相似度')
axes[1].set_ylabel('频数')
axes[1].set_title('类内 vs 类间余弦相似度分布')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

print(f"类内平均相似度: {np.mean(intra_similarities):.4f} ± {np.std(intra_similarities):.4f}")
print(f"类间平均相似度: {np.mean(inter_similarities):.4f} ± {np.std(inter_similarities):.4f}")

plt.tight_layout()
plt.show()


In [22]:
# 可视化：使用t-SNE降维可视化原始数据和筛选后的数据
reducer = TSNE(
    n_components=2,
    perplexity=30,
    metric='cosine',
    random_state=42,
)

# 原始数据可视化
embeddings_2d_original = reducer.fit_transform(selected_embeddings)

# 筛选后的数据可视化（使用马氏距离筛选）
if len(filtered_emb_maha) > 0:
    embeddings_2d_filtered = reducer.fit_transform(filtered_emb_maha)
else:
    embeddings_2d_filtered = embeddings_2d_original

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

unique_labels_sorted = sorted(set(selected_labels))
if len(unique_labels_sorted) <= 10:
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels_sorted)))
else:
    colors = plt.cm.hsv(np.linspace(0, 1, len(unique_labels_sorted)))

label_to_color = dict(zip(unique_labels_sorted, colors))

# 原始数据
for label in unique_labels_sorted:
    mask = selected_labels == label
    axes[0].scatter(
        embeddings_2d_original[mask, 0], 
        embeddings_2d_original[mask, 1],
        c=[label_to_color[label]],
        label=label + f"({np.sum(mask)})",
        alpha=0.6,
        s=20
    )

axes[0].set_title(f'原始数据 t-SNE可视化 (n={len(selected_embeddings)})')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].grid(True, alpha=0.3)

# 筛选后的数据
if len(filtered_emb_maha) > 0:
    for label in unique_labels_sorted:
        mask = filtered_labels_maha == label
        if np.sum(mask) > 0:
            axes[1].scatter(
                embeddings_2d_filtered[mask, 0], 
                embeddings_2d_filtered[mask, 1],
                c=[label_to_color[label]],
                label=label + f"({np.sum(mask)})",
                alpha=0.6,
                s=20
            )

axes[1].set_title(f'马氏距离筛选后 t-SNE可视化 (n={len(filtered_emb_maha)})')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [23]:
# 总结报告
print("=" * 80)
print("高斯分布Embedding分析总结报告")
print("=" * 80)

print(f"\n1. 数据概览:")
print(f"   总样本数: {len(selected_embeddings)}")
print(f"   特征维度: {selected_embeddings.shape[1]}")
print(f"   类别数: {len(filter_model.unique_labels)}")

print(f"\n2. 筛选结果:")
print(f"   马氏距离筛选 (95%分位数): {len(filtered_emb_maha)}/{len(selected_embeddings)} "
      f"({len(filtered_emb_maha)/len(selected_embeddings)*100:.1f}%)")

print(f"\n3. 相似度分析:")
print(f"   类内平均相似度: {np.mean(intra_similarities):.4f} ± {np.std(intra_similarities):.4f}")
print(f"   类间平均相似度: {np.mean(inter_similarities):.4f} ± {np.std(inter_similarities):.4f}")
print(f"   相似度差异: {np.mean(intra_similarities) - np.mean(inter_similarities):.4f}")

print(f"\n4. 各类别质量评估 (按类内相似度排序):")
class_quality = []
for label in sorted(filter_model.unique_labels):
    stat = stats[label]
    class_quality.append((label, stat['mean_intra_cosine_similarity'], stat['n_samples']))

class_quality.sort(key=lambda x: x[1], reverse=True)
for i, (label, sim, n) in enumerate(class_quality, 1):
    print(f"   {i}. {label}: 相似度={sim:.4f}, 样本数={n}")

print("\n" + "=" * 80)
